<a href="https://colab.research.google.com/github/Dharmendra0305/PRODIGY_GA_02/blob/main/Image_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install diffusers transformers accelerate torch gradio

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# -------------------------------
# 1. Load the Stable Diffusion Model
# -------------------------------
# We are using Stable Diffusion v1.5 (industry standard).
# torch_dtype=torch.float16 makes it more memory efficient.
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype = torch.float16)

# -------------------------------
# 2. Move Model to GPU
# -------------------------------
# 'cuda' allows PyTorch to use the GPU for faster inference.
pipe = pipe.to("cuda")

# -------------------------------
# 3. Define Prompts
# -------------------------------
# Positive prompt: what you want to see.
# Negative prompt: what you want to avoid in the image.
prompt = "A highly detailed, futuristic city skyline at sunset, cyberpunk art style, 4k, masterpiece"
negative_prompt = "blurry, low quality, distorted, ugly, text, watermark"

# -------------------------------
# 4. Set Random Seed
# -------------------------------
# Ensures reproducibility: same seed = same output.
generator = torch.Generator("cuda").manual_seed(42)

# -------------------------------
# 5. Generate Image
# -------------------------------
# num_inference_steps: controls quality/detail (higher = more detailed).
# guidance_scale: how strongly the prompt influences the image.
image = pipe( prompt = prompt,
             negative_prompt = negative_prompt,
              num_inference_steps = 50,
              guidance_scale = 7.5,
              generator = generator
              ).images[0]

# -------------------------------
# 6. Save and Display
# -------------------------------
# Save the generated image locally and display inline (Colab/Jupyter).
image.save("my_first_generation.png")
image # This line makes the image appear right inside Colab

In [ ]:
import gradio as gr

# -------------------------------
# 1. Define the Image Generation Function
# -------------------------------
# This function connects the user interface to the Stable Diffusion pipeline.
# It takes user inputs (prompt and negative prompt) and returns a generated image.
def generate_image(prompt, negative_prompt):
  # Generates the image based on user inputs
  image = pipe(prompt, negative_prompt = negative_prompt).images[0]
  return image

# -------------------------------
# 2. Build the Web Interface
# -------------------------------
# Gradio Interface allows us to create a simple web UI.
# Inputs: Two textboxes (prompt and negative prompt).
# Output: An image component to display the generated result.
interface = gr.Interface(
    fn = generate_image,
    inputs = [
        gr.Textbox(label = "Your Prompt", placeholder = "e.g., A futuristic city at sunset..."),
        gr.Textbox(label = "Negative Prompt", placeholder = "e.g., blurry, ugly, distorted...")
    ],
    outputs = gr.Image(label = "Generated Image"),
    title = "My Image Generation Project",
    description = "Built using Stable Diffusion."
)

# -------------------------------
# 3. Launch the Interface
# -------------------------------
# share=True creates a public link that can be accessed from anywhere.
interface.launch(share = True)